# Deploy a LangGraph Agent to Databricks Model Serving

This notebook deploys our CS4603 study assistant agent as a **Databricks Model Serving endpoint**.

**Run this notebook inside your Databricks workspace** (not locally). It uses:
- `mlflow.langchain.log_model()` with models-from-code to package `agent.py`
- `mlflow.register_model()` to publish to Unity Catalog
- `databricks.agents.deploy()` to create the serving endpoint in one call

Once deployed, the endpoint is OpenAI-compatible — call it with `openai.OpenAI` just like any other model.

## Step 0 — Prerequisites

Before running this notebook:

1. Upload `agent.py` to your Databricks workspace (same folder as this notebook)
2. Ensure you have access to Unity Catalog (`main.default` catalog/schema)
3. Your workspace must have Model Serving enabled

The `agent.py` file defines our LangGraph agent with 4 tools. At the bottom it calls
`mlflow.models.set_model(graph)` which tells MLflow which object to serve.

## Step 1 — Install Dependencies

Install the packages that `agent.py` needs. These are also the packages
that will be installed in the serving container.

In [ ]:
%pip install mlflow langchain langgraph langchain-openai langchain-core openai --upgrade 'databricks-sdk>=0.23.0'
dbutils.library.restartPython()

## Step 2 — Configure Environment Variables

The agent needs these to call the LLM model-serving endpoint.
Set them here so `agent.py` can import successfully.

In [ ]:
import os

# These are the same values you'd put in .env locally
os.environ["DATABRICKS_HOST"] = spark.conf.get("spark.databricks.workspaceUrl", "")
if not os.environ["DATABRICKS_HOST"].startswith("https://"):
    os.environ["DATABRICKS_HOST"] = "https://" + os.environ["DATABRICKS_HOST"]

# Use the notebook's token for model serving calls
os.environ["DATABRICKS_TOKEN"] = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
os.environ["DATABRICKS_MODEL"] = "databricks-qwen35-122b-a10b"

print(f"Host:  {os.environ['DATABRICKS_HOST']}")
print(f"Model: {os.environ['DATABRICKS_MODEL']}")

## Step 3 — Verify agent.py Imports

Before logging, let's make sure `agent.py` loads without errors.
If this cell fails, fix the import error before proceeding.

In [ ]:
import importlib.util
import mlflow

# Suppress set_model during test import
_orig = mlflow.models.set_model
mlflow.models.set_model = lambda *a, **kw: None

spec = importlib.util.spec_from_file_location("agent", "agent.py")
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mlflow.models.set_model = _orig

print(f"Agent loaded: {len(mod.tools)} tools")
print(f"Tools: {[t.__name__ for t in mod.tools]}")

## Step 4 — Log the Model to MLflow

This is the key step. We use `mlflow.langchain.log_model()` with **models-from-code**:
we pass the path to `agent.py` (not the graph object). MLflow stores the source file
as an artifact and re-executes it at serving time.

Since we're running on Linux inside Databricks, the path is Linux-native and
the serving container can find it.

In [ ]:
import mlflow
import langchain
import langgraph

mlflow.set_registry_uri("databricks-uc")

experiment_name = f"/Users/{spark.sql('SELECT current_user()').first()[0]}/wk5-deployment"
mlflow.set_experiment(experiment_name)
print(f"Experiment: {experiment_name}")

with mlflow.start_run(run_name="langgraph-agent") as run:
    model_info = mlflow.langchain.log_model(
        lc_model="agent.py",
        name="langgraph_agent",
        input_example={"messages": [{"role": "user", "content": "What is RAG?"}]},
    )

print(f"Model URI: {model_info.model_uri}")
print(f"Run ID:    {run.info.run_id}")

## Step 5 — Register in Unity Catalog

Promote the logged model to Unity Catalog so Model Serving can find it.

In [ ]:
UC_MODEL_NAME = "main.default.cs4603_langgraph_agent"

registered = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=UC_MODEL_NAME,
)
print(f"Registered: {UC_MODEL_NAME} version {registered.version}")

## Step 6 — Deploy the Endpoint

Use the Databricks SDK to create the serving endpoint. The `create_and_wait()` method
provisions the container, loads the model, and waits until the endpoint is READY.

This will take **3-8 minutes** on first deploy. The cell will block until complete.

In [ ]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()
endpoint_name = "cs4603-langgraph-agent"

# Check if endpoint exists and delete if in failed state
try:
    existing = w.serving_endpoints.get(endpoint_name)
    print(f"Endpoint exists with state: {existing.state.config_update}")
    print("Deleting existing endpoint...")
    w.serving_endpoints.delete(endpoint_name)
    import time; time.sleep(5)
    print("Endpoint deleted successfully")
except Exception as e:
    print(f"No existing endpoint found (OK to proceed): {e}")

# Create the endpoint with secret references for agent.py's environment variables.
# agent.py reads DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_MODEL at import time.
print("Creating endpoint...")
w.serving_endpoints.create(
    name=endpoint_name,
    config=EndpointCoreConfigInput(
        name=endpoint_name,
        served_entities=[
            ServedEntityInput(
                entity_name=UC_MODEL_NAME,
                entity_version=str(registered.version),
                workload_size="Small",
                scale_to_zero_enabled=True,
                environment_vars={
                    "DATABRICKS_HOST": "{{secrets/cs4603-deploy/DATABRICKS_HOST}}",
                    "DATABRICKS_TOKEN": "{{secrets/cs4603-deploy/DATABRICKS_TOKEN}}",
                    "DATABRICKS_MODEL": "{{secrets/cs4603-deploy/DATABRICKS_MODEL}}",
                },
            )
        ],
    ),
)

print(f"Endpoint '{endpoint_name}' created successfully.")
print("\nDeployment takes 3-8 minutes. Check Serving UI for status.")
print(f"URL (once READY): {os.environ['DATABRICKS_HOST']}/serving-endpoints/{endpoint_name}/invocations")

## Step 7 — Test the Deployed Endpoint

Once the endpoint shows **READY** in the Serving UI, run this cell to test it.
We use the same `openai.OpenAI` client pattern from the rest of the course.

In [ ]:
import openai
import os

client = openai.OpenAI(
    api_key=os.environ["DATABRICKS_TOKEN"],
    base_url=f"{os.environ['DATABRICKS_HOST']}/serving-endpoints",
)

response = client.chat.completions.create(
    model="cs4603-langgraph-agent",
    messages=[{"role": "user", "content": "Convert 100 degrees Fahrenheit to Celsius"}],
)

final_message = response[0].messages[-1]
print(final_message['content'])

100 degrees Fahrenheit is equal to approximately 37.78 degrees Celsius.


## Cleanup

To delete the endpoint and stop billing:

```python
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
w.serving_endpoints.delete("cs4603-langgraph-agent")
```

With `scale_to_zero=True`, the endpoint costs nothing when idle —
only delete it if you want to fully remove the resource.